In [1]:
import openai
import json
from utils import Spotify
import tiktoken
import time

In [7]:
import os
# os.urandom(24)

from os.path import join, dirname
join(dirname('utils.py'), '.env')

'.env'

In [14]:
os.curdir

'.'

In [2]:
# Load the tokenizer for GPT-4 (gpt-4 and gpt-3.5-turbo use the same encoding)
encoding = tiktoken.encoding_for_model("gpt-4o")
with open('../instance/config.json') as config_file:
    
    config = json.load(config_file)

In [48]:
from spotipy.oauth2 import SpotifyClientCredentials,SpotifyOAuth
# from spotipy.oauth2 import 
import spotipy 
import pandas as pd
FEATURES = [
    'danceability', 'energy', 'acousticness', 'instrumentalness', 'valence', 'loudness', 'tempo',
]
class Spotify:
    """
    ---------------------------------------------------------------------------------------------
    Spotify class helps to extract tracks and its audio features from user playlists
    ---------------------------------------------------------------------------------------------
    Parameters:
        - client_id (str): User client id
        - client_secret (str): User secret id
    ---------------------------------------------------------------------------------------------
    Attributes:
        - client_ (spotipy.Spotify object): Initialized Spotify client object  
        - playlists_name_ (List[str]): List of all playlist names
        - df_ (pandas.DataFrame): Data with tracks and audio features
    ---------------------------------------------------------------------------------------------
    Methods
        - get_tracks_from_playlists: Extract tracks and audio features from user playlists
    ---------------------------------------------------------------------------------------------
    """
    def __init__(self, client_id=None, client_secret=None, redirect_uri=None, auth_token=None, scope = "playlist-read-private playlist-read-collaborative playlist-modify-public user-library-read"):
        self.client_id=client_id
        self.client_secret=client_secret
        self.redirect_uri = redirect_uri
        self.scope = scope
        self.auth_token=auth_token
    
    def connect(self):
        # client_creds = SpotifyClientCredentials(client_id=self.client_id, 
        #                                         client_secret=self.client_secret)
        
        # client = spotipy.Spotify(client_credentials_manager=client_creds)
        # client_creds = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret, redirect_uri=SPOTIFY_REDIRECT_URI)
        if self.auth_token is not None:
            client =  spotipy.Spotify(auth=self.auth_token)
            
        else:
            print('here')
            client = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=self.client_id,
                                                client_secret=self.client_secret,
                                                redirect_uri=self.redirect_uri,
                                                scope=self.scope))
        self.client_ = client


        
    def get_playlist(self) -> pd.DataFrame:
        playlists = []
        offset = 0
        while True:
            response = self.client_.current_user_playlists(offset=offset, limit=50)
            if response is not None:
                playlists.extend(response['items'])
            if response['next']:
                offset += len(response['items'])
            else:
                break
        # print(playlists)
        playlists = [playlist for playlist in playlists if playlist is not None]
        self.playlists_detail = playlists
        self.playlists_name_ = [playlist['name'] for playlist in playlists]
        return pd.DataFrame(playlists)
    
    def _get_tracks_from_playlists(self, playlists, unique):
        track_ls = []
        playlists_ls = self.playlists_name_ if playlists is None else playlists
        
        for playlist in self.playlists_detail:
            name = playlist['name']
            is_public = 1 if playlist['public'] else 0  # Check if the playlist is public
            
            if name not in playlists_ls:
                continue
            
            results = self.client_.playlist(playlist['id'], fields="tracks,next")
            tracks = results['tracks']

            for i, item in enumerate(tracks['items']):
                track_ls.append((name, item['track']['id'], item['track']['name'], is_public))  # Include `is_public`
        
        tracks_df = pd.DataFrame(track_ls, columns=['playlist', 'id', 'name', 'public']).drop_duplicates() \
                if unique else spd.DataFrame(track_ls, columns=['playlist', 'id', 'name', 'public'])
        return tracks_df
    
    
    def _get_audio_features_from_tracks(self, list_of_id, unique):
        tracks_detail = []
        for i in range(len(list_of_id) // 100 + 1):
            tracks_subset = list_of_id[i*100: (i+1)*100]
            audio_features_dict = [x for x in self.client_.audio_features(tracks_subset) if x is not None]
            if (len(tracks_subset) > 0):
                tracks_detail += audio_features_dict

        features_df = pd.DataFrame(tracks_detail).drop_duplicates() if unique \
                      else pd.DataFrame(tracks_detail)
        return features_df
    
    
    def get_tracks_from_playlists(self, limit=50, playlists=None, unique=True):
        """
        ------------------------------------------------------------------------------------------
        get_tracks_from_playlists connects to spotify api and extracts all tracks and 
                                  audio features from playlists
        ------------------------------------------------------------------------------------------
        Parameters:
            - username (str): Spotify username id
            - limit (int): Maximum number of playlists
            - playlists (List[str]): List of playlist names
            - unique (Bool): Returns unique tracks if true
        ------------------------------------------------------------------------------------------
        Effects:
            - Creates df_ attribute
        ------------------------------------------------------------------------------------------
        """
        # self.username = username
        
        # self.get_playlist(username, limit)
        
        self.playlists_df_ = self._get_tracks_from_playlists(playlists, unique)
        song_ids = self.playlists_df_.id.astype(str)
        features_df = self._get_audio_features_from_tracks(song_ids, unique)
        self.songs_df_ = pd.merge(self.playlists_df_, features_df, how='left', left_on='id', right_on='id')

    
    def get_df(self) -> pd.DataFrame:
        cols = ['name'] + FEATURES
        return self.songs_df_[cols].copy()
    


In [52]:
# Obtain user credentials
client_id = config['client_id']
client_secret = config['client_secret']
redirect_uri = config['redirect_uri']

# Initialize
sp = Spotify(client_id, client_secret, redirect_uri)

# Connect
sp.connect()

# Get list of playlist and allow user to select which playlists to extract songs from
playlist_df = sp.get_playlist()


# # Get all tracks from the selected playlist
sp.get_tracks_from_playlists()
df = sp.get_df()


here


OSError: [Errno 48] Address already in use

In [ ]:
playlist_df

In [40]:
playlists=[]
offset=0
while True:
    response = sp.client_.current_user_playlists(offset=offset, limit=50)
    # items = [print(item['name']) for item in response['items'] if item is not None]
    # print(response['items'])
    # print('\n\n')
    playlists.extend(response['items'])
    if response['next']:
        offset += len(response['items'])
    else:
        break
        # print(playlists)
        # self.playlists_detail = playlists
playlists = [playlist for playlist in playlists if playlist is not None]
# sp.client_.current_user_playlists(offset=0, limit=50)

In [50]:
# !rm .cache
# test = sp.songs_df_.copy()
# test[test['name'].str.contains('Broke In A Minute')]

In [89]:
test = df.copy()
# test[['playlist','public']].drop_duplicates().sort_values(['public','playlist'])
test[test['name'].str.contains('Money')]

,name,danceability,energy,acousticness,instrumentalness,valence,loudness,tempo
319,Money Trees,0.716,0.531,0.0703,0.0,0.344,-7.355,71.994
704,Money Trees,0.716,0.531,0.0709,0.0,0.335,-7.355,71.972


In [18]:
# col_name_dict = {
#     'danceability': 'dance'
#     'acousticness': 'acoustic',
#     'instrumentalness': 'instrum',
#     # 'duration_ms': 'duration',
#     # 'time_signature': 'timesig',
#     # 'speechiness': 'speech',
#     # 'loudness': 'loud',
# }
def count_tokens(messages):
    total_tokens = 0
    for message in messages:
        # Role adds an extra token for each message
        role_token_count = len(encoding.encode(message['role']))
        # Content token count
        content_token_count = len(encoding.encode(message['content']))
        # Add both role and content tokens to total count
        total_tokens += role_token_count + content_token_count + 2  # +2 for separators (message overhead)
    return total_tokens

# OpenAI

In [7]:
# Things to do to reduce tokens
## Round decimals. So they might have similar tokens
## Feature selection
## Change feature names. i.e. accoustincess to accoustic
## Batching:
##    - For each iteration, identify tracks that might be a part of the playlist, and aggregate the result

# Things to add
## A temperature (0 - 1) mark to allow users to tell how closely related should a track be to be a part of the playlist.
## For example: If temperature is closer to 1 then more tracks will be added as the model is allowed to be more creative.
##              If temperature is closer to 0, then less tracks will be added as model less creative.
## Distribution or skewness of temperature??

## Add more details on how the songs should be chosen, i.e. imagine you are going through all your songs saved in your spotify playlist to create another playlist titled ...
## Maybe add more descriptive words like feels and mood 
## Maybe try to give a small example.
## Instead of temperature being a continuous value, maybe have it as a discrete value 1,2,3 whre the prompt will change bsaed on the value.
## Maybe have another variable (flexibility), which includes flexibility to include songs from a wider range of genre

In [31]:
# Based on playlist name
from sklearn.utils import shuffle

# def get_prompt_with_music_features():
    


def generate_playlist_from_name(playlist_name, temperature, df):
    openai.api_key = config['openai_api_key']
    system_content = """
    You are a spotify music playlist curator who organizes tracks into thematic playlists based on their audio features extracted
    from spotify api. 
    - The inputs will be given in the following format:
      playlist name: ...
      temperature: ...
      data: ```...```
    - The data in csv format will be supplied and the features below will be given for each track. These features can be helpful 
      in clustering the tracks into playlists but not they need not be used. 
    - The playlist name will be the theme of the playlist. Based on the playlist name, extract tracks in the data that matches
    the theme and respond with the list of tracks that should belong in the playlist.
    - The temperature parameter is similar to that of the temperature parameter for LLMs. It ranges from 0 to 1, where 1 is creative and 0 is not creative.
      This parameter tells the model how creative it should be when determining if a track belongs to the playlist theme. Naturally, higher temperature should
      mean more tracks are added to the playlist. 
    - The features in the data will be:
        Danceability: Measures how suitable a track is for dancing, based on rhythm and tempo. (Range: 0 to 1)
        Energy: Intensity and activity of a track. Higher values represent more energetic tracks. (Range: 0 to 1)
        Acousticness: Likelihood that a track is acoustic. (Range: 0 to 1)
        Instrumentalness: Measures the likelihood of no vocals. Higher means more instrumental. (Range: 0 to 1)
        Valence: Positiveness or happiness of a track. Higher values sound more positive. (Range: 0 to 1)
        Loudness: Average volume in decibels (dB). (Range: ~ -60 to 0 dB)
        Tempo: Speed of the track, measured in beats per minute (BPM). (Range: 0 to 300+ BPM)

    The response should only be the list of song separated by comma. i.e. songA, songB, songC, ... and so on. Note that songs needs to be related to the theme.
    For example, if there is no afro beats songs in the list of songs given, then no songs can be returned. In the case where no songs are returned, return an empty string.
    """
    prompt = f"""
    playlist name: {playlist_name}
    temperature: {temperature}
    data: {df.to_string(na_rep='NA')}
    """
    messages=[
        {"role": "system", "content": system_content},
        {"role": "user", "content": prompt},
    ]
    # print(prompt)
    print(f"Number of tokens number {len(encoding.encode(system_content)) + len(encoding.encode(prompt))}")
    print(f"Number of tokens: {count_tokens(messages)}")
    try:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature= 0.7
        )
    except openai.RateLimitError:
        raise Exception("Please try again in a few minutes. If this problem persists, contact ...")
        
    return response.choices[0].message.content
df = sp.get_df()
df = shuffle(df)
df = sp.get_df()
df = df.loc[5:150, ~df.columns.isin(['playlist', 'id'])]
start = time.monotonic()
string = generate_playlist_from_name('Morning drive', 0.5, df)
print(string)
print(f"Time taken: {round(time.monotonic()-start, 4)} seconds")

Number of tokens number 7367
Number of tokens: 7373
I KNOW ?, BUTTERFLY EFFECT, No Idea, CAN'T SAY, Space Cadet (feat. Gunna), Hot (Remix) [feat. Gunna and Travis Scott], Cardigan, OUT WEST (feat. Young Thug), Wake Up in the Sky, For The Night (feat. Lil Baby & DaBaby), Life Is Good (feat. Drake), Blueberry Faygo, Laugh Now Cry Later (feat. Lil Durk), Lemonade (feat. NAV), FRANCHISE (feat. Young Thug & M.I.A.), No Role Modelz, Ric Flair Drip (with Metro Boomin), goosebumps, Drip Too Hard (Lil Baby & Gunna), Best You Had, Love 119, Haven't Met You Yet, Redbone, Wake Up in the Sky, Pumped Up Kicks
Time taken: 5.2158 seconds


# Gemini

In [20]:
%pip install -U -q "google-generativeai>=0.8.3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-storage 1.31.0 requires google-auth<2.0dev,>=1.11.0, but you have google-auth 2.36.0 which is incompatible.
google-cloud-core 1.7.1 requires google-api-core<2.0.0dev,>=1.21.0, but you have google-api-core 2.23.0 which is incompatible.
google-cloud-core 1.7.1 requires google-auth<2.0dev,>=1.24.0, but you have google-auth 2.36.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [21]:
import google.generativeai as genai
# from IPython.display import HTML, Markdown, display

In [22]:
GOOGLE_API_KEY = ""
genai.configure(api_key=GOOGLE_API_KEY)

In [52]:
def get_prompt(df, playlist_name, temperature):
  prompt = f"""
      You are a spotify music playlist curator who organizes tracks into thematic playlists based on their audio features extracted
      from spotify api. 
      - The inputs will be given in the following format:
        playlist name: ...
        temperature: ...
        data: ```...```
      - The data in csv format will be supplied and the features below will be given for each track. These features can be helpful 
        in clustering the tracks into playlists but not they need not be used. 
      - The playlist name will be the theme of the playlist. Based on the playlist name, only extract tracks in 'data' that matches
      the theme and respond with the list of tracks that should belong in the playlist.
      - The temperature parameter is similar to that of the temperature parameter for LLMs. It ranges from 0 to 1, where 1 is creative and 0 is not creative.
        This parameter tells the model how creative it should be when determining if a track belongs to the playlist theme. Naturally, higher temperature should
        mean more tracks are added to the playlist. 
      - The features in the data will be:
          Danceability: Measures how suitable a track is for dancing, based on rhythm and tempo. (Range: 0 to 1)
          Energy: Intensity and activity of a track. Higher values represent more energetic tracks. (Range: 0 to 1)
          Acousticness: Likelihood that a track is acoustic. (Range: 0 to 1)
          Instrumentalness: Measures the likelihood of no vocals. Higher means more instrumental. (Range: 0 to 1)
          Valence: Positiveness or happiness of a track. Higher values sound more positive. (Range: 0 to 1)
          Loudness: Average volume in decibels (dB). (Range: ~ -60 to 0 dB)
          Tempo: Speed of the track, measured in beats per minute (BPM). (Range: 0 to 300+ BPM)
      - Remember to only extract songs included in 'data'
      Note that songs needs to be related to the theme. For example, if there is no afro beats songs in the list of songs given, then no songs can be returned. In the case where no songs are returned, return an empty string.
      Refer to the inputs below:
      playlist name: {playlist_name}
      temperature: {temperature}
      data: {df.to_string(na_rep='NA')}

      The response should only be the list of song separated by comma. i.e. songA, songB, songC, ... 
      """
  return prompt
df = sp.get_df().drop_duplicates()
prompt = get_prompt(df, 'Morning drive', 0.5)

In [84]:
flash.count_tokens(prompt)

total_tokens: 63182

In [85]:
print(f"Number of tokens: {len(encoding.encode(prompt))}")

Number of tokens: 44857


In [54]:
flash = genai.GenerativeModel('gemini-1.5-flash')
chat = flash.start_chat(history=[])
# response = chat.generate_content(prompt)
response = chat.send_message(prompt)
print(response.text)

Father Stretch My Hands Pt. 1, HIGHEST IN THE ROOM, CAN'T SAY, HOLIDAY, OUT WEST (feat. Young Thug), I KNOW ?, BUTTERFLY EFFECT, When The Swallows Come Back To Capistrano, It's Funny To Everyone But Me, I'm Beginning To See The Light, Someone's Rocking My Dreamboat, Java Jive, You Brought a New Kind of Love to Me, Asleep Among Endives, No Idea, WHAT TO DO? (feat. Don Toliver), Space Cadet (feat. Gunna), Hot (Remix) [feat. Gunna and Travis Scott], 20 Min, Cardigan, Fair Trade (with Travis Scott), Broke In A Minute, Rich Nigga Shit (feat. Young Thug), Wake Up in the Sky, After Party, 5% TINT, Time Flies, Overdue (with Travis Scott), THE SCOTTS, ball w/o you, Had Enough (feat. Quavo & Offset), Life Is Good (feat. Drake), Chicago Freestyle (feat. Giveon), Excitement, Runnin, For The Night (feat. Lil Baby & DaBaby), NO BYSTANDERS, No Role Modelz, Blueberry Faygo, Laugh Now Cry Later (feat. Lil Durk), ORANGE SODA, Lemonade (feat. NAV), FRANCHISE (feat. Young Thug & M.I.A.), GANG GANG, Pain 1

In [64]:
response = chat.send_message('Now with the songs data provided earlier, create a playlist based on the following prompt: Work out bros')
print(response.text)

Life Is Good (feat. Drake),  Praise God,  BOP,  Ric Flair Drip (with Metro Boomin),  Had Enough (feat. Quavo & Offset),  Broke In A Minute,  Rich Nigga Shit (feat. Young Thug),  Wake Up in the Sky,  GANG GANG,  a lot,  Laugh Now Cry Later (feat. Lil Durk),  FRANCHISE (feat. Young Thug & M.I.A.),  20 Min,  NO BYSTANDERS,  Drip Too Hard (Lil Baby & Gunna),  Jimmy Cooks (feat. 21 Savage),  Knife Talk (with 21 Savage ft. Project Pat),  Off The Grid,  Cut Em In (feat. Rick Ross),  Rich Flex,  Cash In Cash Out,  Run It (feat. Rick Ross & Rich Brian),  Family Ties (with Kendrick Lamar),  Just In Time (feat. Lil Wayne & Kenny Mason),  All My Girls Like To Fight,  Sushi For Breakfast,  D.$.M - Mixtape Edit,  Medallion,  SPINBACK,  STEW,  bag or die,  LA Leakers Freestyle,  Purple Takis,  Again,  OG!,  Rich Rich,  Get Back,  Euro$tep,  DNA.,  HUMBLE.,  Marigolds,  Vapor Rub,  Blossom,  SECRETS, Don't, Universe, Ride, If You Let Me, Juice, Cherry Wine, Bad Side, tonight, Yeah You (Thinkin Bout Yo

In [90]:
response.text

"Life Is Good (feat. Drake),  Praise God,  BOP,  Ric Flair Drip (with Metro Boomin),  Had Enough (feat. Quavo & Offset),  Broke In A Minute,  Rich Nigga Shit (feat. Young Thug),  Wake Up in the Sky,  GANG GANG,  a lot,  Laugh Now Cry Later (feat. Lil Durk),  FRANCHISE (feat. Young Thug & M.I.A.),  20 Min,  NO BYSTANDERS,  Drip Too Hard (Lil Baby & Gunna),  Jimmy Cooks (feat. 21 Savage),  Knife Talk (with 21 Savage ft. Project Pat),  Off The Grid,  Cut Em In (feat. Rick Ross),  Rich Flex,  Cash In Cash Out,  Run It (feat. Rick Ross & Rich Brian),  Family Ties (with Kendrick Lamar),  Just In Time (feat. Lil Wayne & Kenny Mason),  All My Girls Like To Fight,  Sushi For Breakfast,  D.$.M - Mixtape Edit,  Medallion,  SPINBACK,  STEW,  bag or die,  LA Leakers Freestyle,  Purple Takis,  Again,  OG!,  Rich Rich,  Get Back,  Euro$tep,  DNA.,  HUMBLE.,  Marigolds,  Vapor Rub,  Blossom,  SECRETS, Don't, Universe, Ride, If You Let Me, Juice, Cherry Wine, Bad Side, tonight, Yeah You (Thinkin Bout Y

# Add to Spotify account

In [ ]:
# Problem:
# - Check if only one user is able to start_chat at a time.

In [81]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# Set up Spotipy credentials
# SPOTIFY_CLIENT_ID = 'your_client_id'  # Replace with your Spotify client ID
# SPOTIFY_CLIENT_SECRET = 'your_client_secret'  # Replace with your Spotify client secret
SPOTIFY_REDIRECT_URI = 'http://localhost:8080/callback'  # This should match your app's redirect URI

# Authenticate and get token
scope = "playlist-modify-public"
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                               client_secret=client_secret,
                                               redirect_uri=SPOTIFY_REDIRECT_URI,
                                               scope=scope))

# User information
user_id = sp.current_user()["id"]

def create_playlist(playlist_name, song_list):
    # Step 1: Create a public playlist
    playlist = sp.user_playlist_create(user=user_id, name=playlist_name, public=True)
    playlist_id = playlist['id']

    # Step 2: Search for each song and add to the playlist
    track_ids = []
    for song in song_list:
        results = sp.search(q=song, type='track', limit=1)
        if results['tracks']['items']:
            track_ids.append(results['tracks']['items'][0]['id'])

    if track_ids:
        sp.playlist_add_items(playlist_id, track_ids)
        print(f"Playlist '{playlist_name}' created with {len(track_ids)} songs!")
    else:
        print("No valid tracks found to add to the playlist.")

if __name__ == "__main__":
    # Input: List of songs
    song_string = response.text
    songs = [song.strip() for song in song_string.split(",")]

    # Create a playlist
    playlist_name = "My New Playlist"  # Change this to your preferred playlist name
    create_playlist(playlist_name, songs)


Playlist 'My New Playlist' created with 57 songs!
